In [1]:
from mendrive_ctypes import lib, MenDriveCpp

In [2]:
# Ячейка: перенесённая логика main() -- скан, FFT, сходимость, гистерезис, сравнение
import numpy as np
import matplotlib.pyplot as plt
import time, os

def shoelace_area(x, y):
    """Площадь замкнутой кривой (проверка петли гистерезиса)."""
    return 0.5*abs(np.sum(x*np.roll(y,-1) - np.roll(x,-1)*y))

def scan_resonance_cpp(N, freqs, ferrite_model='JA', n_sub=1, n_fp=1, n_newton=3, **kwargs):
    resp = []
    for w in freqs:
        sim = MenDriveCpp(N, ferrite_model=ferrite_model, n_sub=n_sub, n_fp=n_fp, n_newton=n_newton, **kwargs)
        res = sim.run(w, n_periods=3, record_from_period=1, amp=0.3, ramp_periods=1)
        a_resp = (res['Hn'].max()-res['Hn'].min())/2 if (len(res['Hn'])>0 and not res['blew_up']) else np.nan
        resp.append(a_resp)
    return np.array(resp)

def main_cpp(N=20, ferrite_model='Preisach2D', bias_orientation='x', H0_bias=4.0,
             excitation_mode='magnetic_right', sigma_e_left=3.0, sigma_m_leak=3.0,
             Ms=1.0, a_JA=0.3, alpha_JA=0.001, k_JA=0.15, c_JA=0.15,
             Ms_llg=0.3, gamma_llg=1.0, alpha_llg=0.1,
             n_hyst_pr=14,
             n_sub=2, n_fp=2, n_newton=4,
             freqs_wide=None, freqs_fine_halfwidth=0.9, freqs_fine_step=0.1,
             n_periods_total=80, record_from_period=15, amp=1.0, ramp_periods=2.0,
             probe_idx=0, last_frac_force=0.5,
             make_plots=True, out_dir='./outputs', show_plots=False, verbose=True):
    """Полный порт main() на C++ ядро вместо чистого Python. См. docstring
    в предыдущем ответе для подробного описания шагов и возвращаемых ключей."""
    os.makedirs(out_dir, exist_ok=True)
    t_start = time.time()
    common_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                          Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                          n_hyst_pr=n_hyst_pr,
                          Ms=Ms, a_JA=a_JA, alpha_JA=alpha_JA, k_JA=k_JA, c_JA=c_JA,
                          sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                          excitation_mode=excitation_mode)
    plots = []

    if freqs_wide is None:
        freqs_wide = np.arange(1.0, 16.01, 0.5)
    resp_wide = scan_resonance_cpp(N, freqs_wide, ferrite_model=ferrite_model,
                                    n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    omega_coarse = freqs_wide[np.nanargmax(resp_wide)]
    if verbose:
        print(f"[main_cpp] Грубый скан: пик omega0~{omega_coarse:.2f}, {time.time()-t_start:.2f}с")

    lo = max(freqs_wide[0], omega_coarse - freqs_fine_halfwidth)
    hi = omega_coarse + freqs_fine_halfwidth
    freqs_fine = np.arange(lo, hi + 1e-9, freqs_fine_step)
    resp_fine = scan_resonance_cpp(N, freqs_fine, ferrite_model=ferrite_model,
                                    n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    omega_res = freqs_fine[np.nanargmax(resp_fine)] if not np.all(np.isnan(resp_fine)) else omega_coarse
    if verbose:
        print(f"[main_cpp] Уточнённый резонанс: omega0={omega_res:.3f}, {time.time()-t_start:.2f}с")

    if make_plots:
        plt.figure(figsize=(12,6))
        plt.plot(freqs_wide, resp_wide, 'o--', ms=3, alpha=0.5, label='грубый скан')
        plt.plot(freqs_fine, resp_fine, 'o-', ms=4, color='tab:blue', label='уточняющий скан')
        plt.axvline(omega_res, color='red', ls='--', label=f'omega0={omega_res:.2f}')
        plt.xlabel('omega0'); plt.ylabel('амплитуда отклика Hz')
        plt.title(f'{ferrite_model} {excitation_mode} amp={amp} newton={n_newton} n_hyst_pr={n_hyst_pr} $\\sigma_e$={sigma_e_left:.1f} $\\sigma_m$={sigma_m_leak:.1f}: скан резонанса')
        plt.legend(); plt.grid(True); plt.tight_layout()
        p = os.path.join(out_dir, f'scan_{ferrite_model}_{excitation_mode}_amp={amp}_newton={n_newton}_n_hyst_pr={n_hyst_pr}_sigma_el={sigma_e_left}_sigma_m={sigma_m_leak}.png')
        plt.savefig(p, dpi=120);
        if show_plots:
            plt.show();
        plt.close(); plots.append(p)

    sim = MenDriveCpp(N, ferrite_model=ferrite_model, n_sub=n_sub, n_fp=n_fp,
                       n_newton=n_newton, **common_kwargs)
    res_long = sim.run(omega_res, n_periods=n_periods_total,
                        record_from_period=record_from_period, amp=amp,
                        ramp_periods=ramp_periods, probe_idx=probe_idx)
    if verbose:
        n_cov = len(res_long['t'])*res_long['dt']/res_long['T'] if len(res_long['t'])>0 else 0.0
        print(f"[main_cpp] Длинный прогон: {time.time()-t_start:.2f}с, точек={len(res_long['t'])}, "
              f"blew_up={res_long['blew_up']}, периодов записи~{n_cov:.1f}")

    res_llg_ref = None
    if ferrite_model == 'Hybrid':
        llg_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                           Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                           sigma_e_left=sigma_e_left, excitation_mode=excitation_mode)
        sim_llg = MenDriveCpp(N, ferrite_model='LLG', **llg_kwargs)
        res_llg_ref = sim_llg.run(omega_res, n_periods=n_periods_total,
                                   record_from_period=record_from_period, amp=amp,
                                   ramp_periods=ramp_periods, probe_idx=probe_idx)

    t_h, dTxx_h, dt_h, T_h = res_long['t'], res_long['dTxx'], res_long['dt'], res_long['T']
    fft_freqs, fft_mag = np.array([]), np.array([])
    if len(dTxx_h) >= 8:
        Nfft = len(dTxx_h)
        window = np.hanning(Nfft)
        spec = np.fft.rfft(dTxx_h * window)
        fft_freqs = np.fft.rfftfreq(Nfft, d=dt_h) * 2*np.pi
        fft_mag = np.abs(spec)
        if make_plots:
            plt.figure(figsize=(10,6))
            plt.plot(fft_freqs, fft_mag, lw=1.0)
            plt.axvline(omega_res, color='red', ls='--', alpha=0.6, label=f'omega0={omega_res:.2f}')
            if bias_orientation in ('x','y','z') and ferrite_model in ('LLG','Hybrid'):
                plt.axvline(gamma_llg*H0_bias, color='green', ls='--', alpha=0.6,
                            label=f'gamma*H0={gamma_llg*H0_bias:.2f}')
            plt.xlim(0, min(fft_freqs.max(), 4*omega_res))
            plt.xlabel('omega'); plt.ylabel('|FFT(dTxx)|')
            plt.title(f'{ferrite_model} {excitation_mode} amp={amp} newton={n_newton} n_hyst_pr={n_hyst_pr} $\\sigma_e$={sigma_e_left:.1f} $\\sigma_m$={sigma_m_leak:.1f}: спектр dTxx')
            plt.legend(); plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, f'fft_dTxx_{ferrite_model}_{excitation_mode}_amp={amp}_newton={n_newton}_n_hyst_pr={n_hyst_pr}_sigma_el={sigma_e_left:.1f}_sigma_m={sigma_m_leak:.1f}.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    convergence_ratio_cum = np.array([])
    spp_h = int(round(T_h/dt_h)) if len(t_h) > 0 else 0
    if spp_h > 0:
        n_rec_periods = len(t_h) // spp_h
        if n_rec_periods >= 1:
            pm_dTxx = np.array([dTxx_h[i*spp_h:(i+1)*spp_h].mean() for i in range(n_rec_periods)])
            pm_P = np.array([res_long['P'][i*spp_h:(i+1)*spp_h].mean() for i in range(n_rec_periods)])
            cum_F = np.cumsum(pm_dTxx) / np.arange(1, n_rec_periods+1)
            cum_P = np.cumsum(pm_P) / np.arange(1, n_rec_periods+1)
            convergence_ratio_cum = np.where(np.abs(cum_P) > 1e-30, cum_F/cum_P, np.nan)
            if make_plots:
                plt.figure(figsize=(9,4))
                plt.plot(np.arange(1, n_rec_periods+1), convergence_ratio_cum, 'o-', ms=3)
                plt.xlabel('периодов усреднено'); plt.ylabel('накопл. среднее dTxx/P')
                plt.title(f'{ferrite_model} {excitation_mode} amp={amp} newton={n_newton} n_hyst_pr={n_hyst_pr} $\\sigma_e$={sigma_e_left:.1f} $\\sigma_m$={sigma_m_leak:.1f}: сходимость force/power')
                plt.grid(True); plt.tight_layout()
                p = os.path.join(out_dir, f'convergence_{ferrite_model}_{excitation_mode}_amp={amp}_newton={n_newton}_n_hyst_pr={n_hyst_pr}_sigma_el={sigma_e_left:.1f}_sigma_m={sigma_m_leak:.1f}.png')
                plt.savefig(p, dpi=120);
                if show_plots:
                    plt.show();
                plt.close(); plots.append(p)

    hysteresis_area = None
    if ferrite_model == 'Hybrid' and spp_h > 0 and len(res_long['HzJA']) >= spp_h:
        Hloop = res_long['HzJA'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['MzJA'][-spp_h:]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(8,8))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:green', marker='o', ms=2)
            plt.xlabel('H (Э)'); plt.ylabel('B (Гс)')
            plt.title(f'Hybrid: внутренняя петля JA (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, 'hysteresis_Hybrid_internalJA_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)
    elif ferrite_model in ['JA', 'Preisach', 'Preisach2D'] and spp_h > 0:
        Hloop = res_long['Hn'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['Mn'][-spp_h:, 1]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(12,8))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:purple', marker='o', ms=2)
            plt.xlabel('H (Э)'); plt.ylabel('B (Гс)')
            plt.title(f'{ferrite_model} {excitation_mode} amp={amp} newton={n_newton} n_hyst_pr={n_hyst_pr} $\\sigma_e$={sigma_e_left:.1f} $\\sigma_m$={sigma_m_leak:.1f}: петля гистерезиса (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, f'hysteresis_{ferrite_model}_{excitation_mode}_amp={amp}_newton={n_newton}_n_hyst_pr={n_hyst_pr}_sigma_el={sigma_e_left:.1f}_sigma_m={sigma_m_leak:.1f}.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    if ferrite_model == 'Hybrid' and res_llg_ref is not None and spp_h > 0:
        n_traj = min(3*spp_h, len(res_long['Mn']), len(res_llg_ref['Mn']))
        if n_traj > 0:
            fig, axes = plt.subplots(1, 2, figsize=(12,5))
            axes[0].plot(res_long['Mn'][-n_traj:,1], res_long['Mn'][-n_traj:,2], lw=0.8, color='tab:green')
            axes[0].set_title('Hybrid: траектория M'); axes[0].set_aspect('equal'); axes[0].grid(True)
            axes[1].plot(res_llg_ref['Mn'][-n_traj:,1], res_llg_ref['Mn'][-n_traj:,2], lw=0.8, color='tab:orange')
            axes[1].set_title('Чистый LLG: траектория M'); axes[1].set_aspect('equal'); axes[1].grid(True)
            plt.tight_layout()
            p = os.path.join(out_dir, 'M_trajectory_Hybrid_vs_LLG_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    force_per_power_code, force_per_kW = sim.force_per_power(res_long, last_frac=last_frac_force)
    if verbose:
        print(f"[main_cpp] FORCE/POWER: {force_per_power_code:.4e} (код.ед.), {force_per_kW:.4e} Н/кВт")

    P_avg, Pdiss_avg, rel_dif = sim.check_energy_balance(res_long, last_frac=last_frac_force)
    if verbose:
        print(f"[main_cpp] check_energy_balance: P_avg = {P_avg:.4e}, Pdiss_avg={Pdiss_avg:.4e} rel_dif={rel_dif}")
        print(f"[main_cpp] Итого времени: {time.time()-t_start:.2f}с")

    return dict(omega_res=omega_res, freqs_wide=freqs_wide, resp_wide=resp_wide,
                freqs_fine=freqs_fine, resp_fine=resp_fine,
                res_long=res_long, res_long_llg_ref=res_llg_ref,
                hysteresis_area=hysteresis_area, fft_freqs=fft_freqs, fft_mag=fft_mag,
                convergence_ratio_cum=convergence_ratio_cum,
                force_per_power_code=force_per_power_code, force_per_kW=force_per_kW, plots=plots)

In [3]:
N = 160
n_newton = 4
ferrite_model='Preisach'
n_hyst_pr=2001
sigma_m_leak=3.0
sigma_e_left=3.0
amp_m = 1
amp_e = 1
n_periods_total=8000
record_from_period=1000

In [4]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                   n_hyst_pr=n_hyst_pr,
                   n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 1*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.55с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.10с
[main_cpp] Длинный прогон: 27385.24с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 5.7238e-01 (код.ед.), 5.7238e+04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.4458e+00, Pdiss_avg=1.9083e+02 rel_dif=0.9819426199724713
[main_cpp] Итого времени: 27395.90с
omega_res = 3.000000000000001
force_per_kW = 57237.868335906875
hysteresis_area = 17018.507752251848


In [5]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 2*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.68с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.20с
[main_cpp] Длинный прогон: 27304.90с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.5820e-01 (код.ед.), 1.5820e+04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.2467e+01, Pdiss_avg=1.9080e+02 rel_dif=0.934662345771922
[main_cpp] Итого времени: 27316.36с
omega_res = 3.000000000000001
force_per_kW = 15820.04306845888
hysteresis_area = 15787.94234869309


In [6]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 3*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.55с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.10с
[main_cpp] Длинный прогон: 27368.72с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 6.5336e-02 (код.ед.), 6.5336e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.0183e+01, Pdiss_avg=1.9079e+02 rel_dif=0.8417989415659284
[main_cpp] Итого времени: 27380.20с
omega_res = 3.000000000000001
force_per_kW = 6533.595639090366
hysteresis_area = 16498.553670626272


In [7]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 4*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.41с
[main_cpp] Уточнённый резонанс: omega0=3.000, 290.83с
[main_cpp] Длинный прогон: 27375.15с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 3.7893e-02 (код.ед.), 3.7893e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 5.1883e+01, Pdiss_avg=1.9078e+02 rel_dif=0.7280533458720847
[main_cpp] Итого времени: 27385.58с
omega_res = 3.000000000000001
force_per_kW = 3789.339680937429
hysteresis_area = 14675.997134497698


In [8]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 5*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.59с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.22с
[main_cpp] Длинный прогон: 27322.88с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 2.4275e-02 (код.ед.), 2.4275e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 8.1315e+01, Pdiss_avg=1.9081e+02 rel_dif=0.57383527746355
[main_cpp] Итого времени: 27333.28с
omega_res = 3.000000000000001
force_per_kW = 2427.476244244809
hysteresis_area = 15539.957864679087


In [9]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 6*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.49с
[main_cpp] Уточнённый резонанс: omega0=3.000, 290.91с
[main_cpp] Длинный прогон: 27353.90с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.6848e-02 (код.ед.), 1.6848e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.1698e+02, Pdiss_avg=1.9077e+02 rel_dif=0.38677807471017733
[main_cpp] Итого времени: 27364.68с
omega_res = 3.000000000000001
force_per_kW = 1684.8340543220477
hysteresis_area = 15354.240475171366


In [10]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 7*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.95с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.44с
[main_cpp] Длинный прогон: 27280.01с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.2303e-02 (код.ед.), 1.2303e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.6008e+02, Pdiss_avg=1.9081e+02 rel_dif=0.1610594118002424
[main_cpp] Итого времени: 27291.75с
omega_res = 3.000000000000001
force_per_kW = 1230.285416625832
hysteresis_area = 16335.747738888236


In [11]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 8*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.84с
[main_cpp] Уточнённый резонанс: omega0=3.000, 291.60с
[main_cpp] Длинный прогон: 27380.75с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.3416e-03 (код.ед.), 9.3416e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.1119e+02, Pdiss_avg=1.9066e+02 rel_dif=0.09718780739286419
[main_cpp] Итого времени: 27391.07с
omega_res = 3.000000000000001
force_per_kW = 934.1634943763773
hysteresis_area = 16135.326687230168


In [12]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 9*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.35с
[main_cpp] Уточнённый резонанс: omega0=3.000, 290.77с
[main_cpp] Длинный прогон: 27288.18с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 7.3859e-03 (код.ед.), 7.3859e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.6668e+02, Pdiss_avg=1.9076e+02 rel_dif=0.2846882280952325
[main_cpp] Итого времени: 27298.62с
omega_res = 3.000000000000001
force_per_kW = 738.5909650586915
hysteresis_area = 15108.394401027286


In [13]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period,  amp = 10*amp_m)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~3.00, 140.46с
[main_cpp] Уточнённый резонанс: omega0=3.000, 290.92с
[main_cpp] Длинный прогон: 27394.98с, точек=29321531, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 6.0537e-03 (код.ед.), 6.0537e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.2574e+02, Pdiss_avg=1.9077e+02 rel_dif=0.4143601575197564
[main_cpp] Итого времени: 27405.70с
omega_res = 3.000000000000001
force_per_kW = 605.3681865532997
hysteresis_area = 16618.0195418633


In [14]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=1*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.89с
[main_cpp] Уточнённый резонанс: omega0=16.900, 172.11с
[main_cpp] Длинный прогон: 4820.41с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9655e-08 (код.ед.), 9.9655e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 5.1196e+00, Pdiss_avg=2.5840e-03 rel_dif=0.9994952776822918
[main_cpp] Итого времени: 4822.50с
omega_res = 16.89999999999999
force_per_kW = 0.009965454126186386
hysteresis_area = 0.0


In [15]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=2*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.77с
[main_cpp] Уточнённый резонанс: omega0=16.900, 171.95с
[main_cpp] Длинный прогон: 4940.76с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9463e-08 (код.ед.), 9.9463e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.0506e+01, Pdiss_avg=1.0367e-02 rel_dif=0.9994944419464945
[main_cpp] Итого времени: 4942.97с
omega_res = 16.89999999999999
force_per_kW = 0.009946330516928536
hysteresis_area = 0.004771026314382976


In [16]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=3*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.69с
[main_cpp] Уточнённый резонанс: omega0=16.900, 171.84с
[main_cpp] Длинный прогон: 5042.37с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9431e-08 (код.ед.), 9.9431e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 4.6114e+01, Pdiss_avg=2.3350e-02 rel_dif=0.9994936518406409
[main_cpp] Итого времени: 5044.45с
omega_res = 16.89999999999999
force_per_kW = 0.009943101369875392
hysteresis_area = 0.015597888795317871


In [17]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=4*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.78с
[main_cpp] Уточнённый резонанс: omega0=16.900, 171.97с
[main_cpp] Длинный прогон: 5109.01с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.0299e-07 (код.ед.), 1.0299e-02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 8.2020e+01, Pdiss_avg=4.1772e-02 rel_dif=0.9994907117974607
[main_cpp] Итого времени: 5111.08с
omega_res = 16.89999999999999
force_per_kW = 0.010299040079379844
hysteresis_area = 0.08816353590893233


In [18]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=5*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.88с
[main_cpp] Уточнённый резонанс: omega0=16.900, 172.08с
[main_cpp] Длинный прогон: 5163.97с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9711e-08 (код.ед.), 9.9711e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.2825e+02, Pdiss_avg=6.5767e-02 rel_dif=0.9994871834042923
[main_cpp] Итого времени: 5166.07с
omega_res = 16.89999999999999
force_per_kW = 0.009971109091753303
hysteresis_area = 0.2835781319689368


In [19]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=6*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.96с
[main_cpp] Уточнённый резонанс: omega0=16.900, 172.18с
[main_cpp] Длинный прогон: 5228.76с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.0084e-07 (код.ед.), 1.0084e-02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.8488e+02, Pdiss_avg=9.6102e-02 rel_dif=0.9994801909615709
[main_cpp] Итого времени: 5230.91с
omega_res = 16.89999999999999
force_per_kW = 0.010084391018155637
hysteresis_area = 0.7863785065747928


In [20]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=7*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.81с
[main_cpp] Уточнённый резонанс: omega0=16.900, 172.05с
[main_cpp] Длинный прогон: 5305.16с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9277e-08 (код.ед.), 9.9277e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.5209e+02, Pdiss_avg=1.3419e-01 rel_dif=0.9994676904755448
[main_cpp] Итого времени: 5307.24с
omega_res = 16.89999999999999
force_per_kW = 0.009927723975402947
hysteresis_area = 2.1682255124185232


In [21]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=8*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.66с
[main_cpp] Уточнённый резонанс: omega0=16.900, 171.80с
[main_cpp] Длинный прогон: 5382.70с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9963e-08 (код.ед.), 9.9963e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.2987e+02, Pdiss_avg=1.8067e-01 rel_dif=0.9994522952369985
[main_cpp] Итого времени: 5384.77с
omega_res = 16.89999999999999
force_per_kW = 0.009996299839579023
hysteresis_area = 4.028552653787006


In [22]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=9*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.73с
[main_cpp] Уточнённый резонанс: omega0=16.900, 171.97с
[main_cpp] Длинный прогон: 5458.30с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.9665e-08 (код.ед.), 9.9665e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 4.1803e+02, Pdiss_avg=2.3652e-01 rel_dif=0.9994342132525723
[main_cpp] Итого времени: 5460.36с
omega_res = 16.89999999999999
force_per_kW = 0.009966497530366557
hysteresis_area = 6.9840920411612


In [23]:
result = main_cpp(N=N, ferrite_model=ferrite_model, bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=n_newton, sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                  n_hyst_pr=n_hyst_pr,
                  n_periods_total=n_periods_total, record_from_period=record_from_period, amp=10*amp_e)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~16.00, 142.93с
[main_cpp] Уточнённый резонанс: omega0=16.900, 172.14с
[main_cpp] Длинный прогон: 5529.87с, точек=5205005, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.7175e-08 (код.ед.), 9.7175e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 5.1650e+02, Pdiss_avg=3.0260e-01 rel_dif=0.999414121359591
[main_cpp] Итого времени: 5531.95с
omega_res = 16.89999999999999
force_per_kW = 0.009717493729714808
hysteresis_area = 10.511860978689327
